# ORIE 5530 Project: Part II Additional Analysis

Supplementary analyses for Part II, collected from the main notebook (`ORIE_5530_Notebook.ipynb`).
The main notebook contains everything the report's Part II needs; this notebook holds the
supporting and verification analyses.

**Self-contained:** this notebook reloads the data and rebuilds the transition matrices itself.
It expects the same `Data/` folder of August 2026 Citi Bike CSVs as the main notebook.

**Random seed:** `SEED = 0`. Because this notebook draws its own simulations, simulated values
differ slightly from the main notebook's, within Monte Carlo error.

| Section | Analysis | Supports in the report |
|---|---|---|
| A | Simulation including the combined chain | Combined chain as a baseline (Part I Q4) |
| B | Exact vs. simulated probability of becoming unavailable | Verification of Part II Q1 |
| C | Sensitivity to the starting state | "When it might not help" (Part II Q2), Part I Q4 limitations |
| D | Expected minutes unavailable at half capacity | Secondary measure for Part II Q1 |
| E | Best starting level, exact calculation | Verification of the Part II Q2 recommendation |
| F | Time until a station first becomes unavailable | Timing of the morning/evening failures |

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
AM_HOURS = range(6, 11)         # 6:00–10:59
PM_HOURS = range(15, 20)        # 15:00–19:59
HOURS = list(AM_HOURS) + list(PM_HOURS)
stations = ["W 21 St & 6 Ave", "11 Ave & W 41 St", "S 2 St & Kent Ave"]
station_capacities = [63, 48, 44]
WINDOW = 10                     # minutes per step
STEP_MIN = WINDOW

N_STEPS = 30                    # steps per block
N_RUNS = 10_000
N_SHOW = 8
BLOCK_START = {"AM": 6, "PM": 15}
SEED = 0

IMG = Path("images")
IMG.mkdir(exist_ok=True)
name_width = max(len(s) for s in stations)
rng = np.random.default_rng(SEED)

### Load, clean, and filter the data (same steps as the main notebook)

In [ ]:
files = sorted(Path("Data").glob("*.csv"))
df = pd.concat(
    (pd.read_csv(f, parse_dates=["started_at", "ended_at"],
                 dtype={"start_station_id": str, "end_station_id": str}) for f in files),
    ignore_index=True,
)

# Cleaning: rides over 3 hours, negative durations, missing start/end station
ride_time = df["ended_at"] - df["started_at"]
bad = ((ride_time > pd.Timedelta(hours=3)) | (ride_time < pd.Timedelta(0))
       | df["start_station_id"].isna() | df["end_station_id"].isna())
df = df.loc[~bad]

# Keep weekday rides in the AM/PM blocks that start or end at our stations
start_in = (df["started_at"].dt.dayofweek < 5) & df["started_at"].dt.hour.isin(HOURS)
end_in   = (df["ended_at"].dt.dayofweek < 5)   & df["ended_at"].dt.hour.isin(HOURS)
at_ours  = df["start_station_name"].isin(stations) | df["end_station_name"].isin(stations)
df = df.loc[(start_in | end_in) & at_ours]

print(f"rides kept: {len(df)}")
assert len(df) == 39898, "should match the main notebook's filtered data (39898 rides)"

### Net flow and transition matrices (same functions as the main notebook)

In [ ]:
AUG_WEEKDAYS = pd.date_range("2026-08-01", "2026-08-31", freq="B")   # Mon–Fri, 21 days

def step_counts(times, hours, window, days=AUG_WEEKDAYS):
    # Number of events in each `window`-minute step of a block, for every day, in time order.
    block_len = len(hours) * 60
    assert block_len % window == 0, f"window ({window}) must divide the block length ({block_len} min)"
    n_steps = block_len // window

    minutes_in = (times - times.dt.normalize()).dt.total_seconds() / 60 - hours.start * 60
    in_block = (minutes_in >= 0) & (minutes_in < block_len)

    step = (minutes_in[in_block] // window).astype(int)
    date = times[in_block].dt.normalize()
    counts = step.groupby([date, step]).size()

    grid = pd.MultiIndex.from_product([days, range(n_steps)])
    return counts.reindex(grid, fill_value=0).to_numpy()


def net_flow_arrays(df, station, am_hours, pm_hours, window, days=AUG_WEEKDAYS):
    # (am_D, pm_D): arrivals - departures at `station` for every step in each block.
    dep_times = df.loc[df["start_station_name"] == station, "started_at"]
    arr_times = df.loc[df["end_station_name"] == station, "ended_at"]
    am_D = step_counts(arr_times, am_hours, window, days) - step_counts(dep_times, am_hours, window, days)
    pm_D = step_counts(arr_times, pm_hours, window, days) - step_counts(dep_times, pm_hours, window, days)
    return am_D, pm_D


def build_transition_matrix(D, capacity):
    # P[i, j] = P(D = j - i), with moves past 0 or capacity counted as moves to 0 or capacity.
    D = np.asarray(D)
    values, counts = np.unique(D, return_counts=True)
    probs = counts / counts.sum()

    P = np.zeros((capacity + 1, capacity + 1))
    for i in range(capacity + 1):
        next_states = np.clip(i + values, 0, capacity)
        np.add.at(P[i], next_states, probs)

    assert np.allclose(P.sum(axis=1), 1), "every row should sum to 1"
    return P


MCs = {}
for station, cap in zip(stations, station_capacities):
    am_D, pm_D = net_flow_arrays(df, station, AM_HOURS, PM_HOURS, WINDOW)
    MCs[station] = {
        "AM": build_transition_matrix(am_D, cap),
        "PM": build_transition_matrix(pm_D, cap),
        "Combined": build_transition_matrix(np.concatenate([am_D, pm_D]), cap),
    }
    print(f"{station:<{name_width}} | capacity {cap} | "
          f"mean D AM {am_D.mean():+.2f}, PM {pm_D.mean():+.2f}")

# Main notebook: W 21 St AM +1.82, PM -1.61 | 11 Ave AM -1.55, PM +0.99 | S 2 St AM +0.16, PM -0.01

### Simulation and exact-calculation helpers

`simulate_many_fast` samples the next state from row $X_t$ of the transition matrix, like the main
notebook's `simulate_many`, but advances all runs at once (inverse-CDF sampling), so it is much faster.

The exact helpers use the transition matrix directly:
- `prob_hit_by`: probability of reaching a set of states within the block (target states made absorbing).
- `exact_minutes_all_starts`: expected minutes spent in a set of states, for every starting state,
  using $\mathbb{E}[\text{steps in } S] = \sum_t \mathbb{P}(X_t \in S)$.

In [ ]:
def simulate_many_fast(P, start, n_steps, n_runs, rng):
    # n_runs independent paths X_0..X_n, shape (n_runs, n_steps + 1).
    cdf = P.cumsum(axis=1)
    paths = np.empty((n_runs, n_steps + 1), dtype=int)
    paths[:, 0] = start
    for t in range(n_steps):
        u = rng.random(n_runs)[:, None]
        nxt = (u > cdf[paths[:, t]]).sum(axis=1)        # inverse-CDF draw from row X_t
        paths[:, t + 1] = np.minimum(nxt, len(P) - 1)   # guard against rounding at 1.0
    return paths


def prob_hit_by(P, start, targets, n_steps=N_STEPS):
    # Exact P(chain visits any state in `targets` within n_steps), from `start`.
    P_abs = P.copy()
    for s in targets:
        P_abs[s] = 0
        P_abs[s, s] = 1
    dist = np.zeros(len(P))
    dist[start] = 1
    dist = dist @ np.linalg.matrix_power(P_abs, n_steps)
    return dist[list(targets)].sum()


def exact_minutes_all_starts(P, states, n_steps=N_STEPS, step_min=STEP_MIN):
    # Expected minutes spent in `states` during the block, for every starting state 0..c.
    dist = np.eye(len(P))
    total = np.zeros(len(P))
    for _ in range(n_steps):
        dist = dist @ P
        total += dist[:, states].sum(axis=1)
    return total * step_min


def clock(block, minutes):
    total = BLOCK_START[block] * 60 + minutes
    return f"{int(total // 60)}:{int(total % 60):02d}"

### Baseline simulation: every station starts each block at half capacity

Sections A, B, D, and F use these runs. Half capacity is `cap // 2` (31, 24, and 22 bikes), as in the main notebook.

In [ ]:
sims = {}
for station, cap in zip(stations, station_capacities):
    sims[station] = {block: simulate_many_fast(MCs[station][block], cap // 2, N_STEPS, N_RUNS, rng)
                     for block in ["AM", "PM", "Combined"]}

## A. Simulation including the combined chain

The combined chain pools AM and PM steps into one distribution of $D$, so it has no real time of day;
its panels use the step number on the x-axis. It is shown only as a baseline: its flat mean paths
reflect the AM and PM drifts cancelling, not any real period of the day.

In [ ]:
BLOCKS = ["AM", "PM", "Combined"]
fig, axes = plt.subplots(len(stations), len(BLOCKS), figsize=(16, 10), sharex="col")

for r, (station, cap) in enumerate(zip(stations, station_capacities)):
    for c, block in enumerate(BLOCKS):
        ax = axes[r, c]
        paths = sims[station][block]
        steps = np.arange(N_STEPS + 1)
        x = BLOCK_START[block] + steps / 6 if block in BLOCK_START else steps

        lo, hi = np.percentile(paths, [5, 95], axis=0)
        ax.fill_between(x, lo, hi, color="#2a78d6", alpha=0.15, lw=0, label="5–95% of runs")
        for i, path in enumerate(paths[:N_SHOW]):
            ax.step(x, path, where="post", color="0.55", lw=0.8,
                    label="individual runs" if i == 0 else None)
        ax.plot(x, paths.mean(axis=0), color="#2a78d6", lw=2, label="mean of runs")

        ax.axhline(0, color="0.3", ls="--", lw=0.8)
        ax.axhline(cap, color="0.3", ls="--", lw=0.8)
        ax.set_ylim(-2, cap + 2)
        ax.set_title(f"{station} | {block} (capacity {cap})", fontsize=10)
        ax.grid(axis="y", color="0.9")
        ax.spines[["top", "right"]].set_visible(False)
        if c == 0:
            ax.set_ylabel("Bikes at station")
        if r == len(stations) - 1:
            ax.set_xlabel("Step (10 min)" if block == "Combined" else "Time of day (hour)")

axes[0, 0].legend(loc="upper left", fontsize=8, frameon=False)
fig.suptitle(f"Simulated station inventory including the combined chain "
             f"({N_RUNS:,} runs, start at half capacity)")
fig.tight_layout()
fig.savefig(IMG / "simulation_runs_with_combined.png", dpi=200, bbox_inches="tight")
plt.show()

## B. Exact vs. simulated probability of becoming unavailable

Checks the Part II Q1 simulation against the exact calculation. The exact value makes 0 and capacity
absorbing and computes the probability mass there after 30 steps. With 18 comparisons at 95% confidence,
about one value falling just outside its interval is expected by chance.

In [ ]:
rows = []
for station, cap in zip(stations, station_capacities):
    for block in ["AM", "PM"]:
        P = MCs[station][block]
        after = sims[station][block][:, 1:]                      # states after steps 1..30
        for name, targets in [("empty", [0]), ("full", [cap]), ("unavailable", [0, cap])]:
            hits = np.isin(after, targets).any(axis=1)
            p_sim = hits.mean()
            ci = 1.96 * np.sqrt(p_sim * (1 - p_sim) / N_RUNS)
            p_exact = prob_hit_by(P, cap // 2, targets)
            rows.append({"station": station, "block": block, "event": f"ever {name}",
                         "simulated": p_sim, "±": ci, "exact": p_exact,
                         "within CI": abs(p_sim - p_exact) <= max(ci, 1e-3)})

check = pd.DataFrame(rows)
print(check.round(3).to_string(index=False))
print(f"\n{check['within CI'].sum()} of {len(check)} simulated values are within their 95% interval")

## C. Sensitivity to the starting state

Repeats the Part II Q1 probability for three starting states: near empty (10% of capacity),
half (`cap // 2`), and near full (90%). Shows how strongly the results depend on the starting-state
assumption, and that no single starting level prevents the Manhattan stations from becoming unavailable.

In [ ]:
def start_levels(cap):
    return {"near empty (10%)": int(round(0.10 * cap)),
            "half (50%)": cap // 2,
            "near full (90%)": int(round(0.90 * cap))}

rows = []
start_sims = {}
for station, cap in zip(stations, station_capacities):
    for block in ["AM", "PM"]:
        P = MCs[station][block]
        for label, start in start_levels(cap).items():
            paths = simulate_many_fast(P, start, N_STEPS, N_RUNS, rng)
            start_sims[(station, block, label)] = paths

            after = paths[:, 1:]
            hit_empty = (after == 0).any(axis=1)
            hit_full = (after == cap).any(axis=1)
            p = (hit_empty | hit_full).mean()
            rows.append({"station": station, "block": block, "start": f"{label}: {start}",
                         "P(ever empty)": hit_empty.mean(),
                         "P(ever full)": hit_full.mean(),
                         "P(ever unavailable)": p,
                         "±": 1.96 * np.sqrt(p * (1 - p) / N_RUNS)})

start_results = pd.DataFrame(rows)
print(start_results.round(3).to_string(index=False))

In [ ]:
colors = {"near empty (10%)": "#2a78d6", "half (50%)": "0.45", "near full (90%)": "#d65a2a"}
fig, axes = plt.subplots(len(stations), 2, figsize=(12, 10), sharex="col")

for r, (station, cap) in enumerate(zip(stations, station_capacities)):
    for c, block in enumerate(["AM", "PM"]):
        ax = axes[r, c]
        hours = BLOCK_START[block] + np.arange(N_STEPS + 1) / 6
        for label in colors:
            paths = start_sims[(station, block, label)]
            lo, hi = np.percentile(paths, [5, 95], axis=0)
            ax.fill_between(hours, lo, hi, color=colors[label], alpha=0.10, lw=0)
            ax.plot(hours, paths.mean(axis=0), color=colors[label], lw=2, label=label)
        ax.axhline(0, color="0.3", ls="--", lw=0.8)
        ax.axhline(cap, color="0.3", ls="--", lw=0.8)
        ax.set_ylim(-2, cap + 2)
        ax.set_title(f"{station} | {block} (capacity {cap})", fontsize=10)
        ax.grid(axis="y", color="0.9")
        ax.spines[["top", "right"]].set_visible(False)
        if c == 0:
            ax.set_ylabel("Bikes at station")
        if r == len(stations) - 1:
            ax.set_xlabel("Time of day (hour)")

axes[0, 0].legend(title="Starting state", loc="center right", fontsize=8, frameon=False)
fig.suptitle(f"Mean simulated inventory by starting state ({N_RUNS:,} runs each)")
fig.tight_layout()
fig.savefig(IMG / "simulation_start_states.png", dpi=200, bbox_inches="tight")
plt.show()

## D. Expected minutes unavailable at half capacity

A secondary measure for Part II Q1: how long, on average, each station spends empty or full during a
300-minute block. A station counts as unavailable for a whole 10-minute step when it is at 0 or capacity
at the end of that step. The exact column checks the simulation.

In [ ]:
rows = []
for station, cap in zip(stations, station_capacities):
    for block in ["AM", "PM"]:
        P = MCs[station][block]
        after = sims[station][block][:, 1:]
        min_empty = (after == 0).sum(axis=1) * STEP_MIN
        min_full = (after == cap).sum(axis=1) * STEP_MIN
        min_unavail = min_empty + min_full
        exact = (exact_minutes_all_starts(P, [0]) + exact_minutes_all_starts(P, [cap]))[cap // 2]
        rows.append({"station": station, "block": block,
                     "E[min empty]": min_empty.mean(),
                     "E[min full]": min_full.mean(),
                     "E[min unavailable]": min_unavail.mean(),
                     "±": 1.96 * min_unavail.std(ddof=1) / np.sqrt(N_RUNS),
                     "exact": exact})

time_results = pd.DataFrame(rows)
print(time_results.round(1).to_string(index=False))

## E. Best starting level: exact calculation

Verification for the Part II Q2 recommendation. For every possible starting level, computes the exact
expected minutes the station spends empty or full during the block, and finds the level that minimizes it.
The main notebook finds the best start by simulation; the two should agree to within a bike or two,
since the curve is flat near its minimum.

In [ ]:
fig, axes = plt.subplots(len(stations), 2, figsize=(12, 10))
rows = []

for r, (station, cap) in enumerate(zip(stations, station_capacities)):
    for c, block in enumerate(["AM", "PM"]):
        P = MCs[station][block]
        m_empty = exact_minutes_all_starts(P, [0])
        m_full = exact_minutes_all_starts(P, [cap])
        m_total = m_empty + m_full
        best = int(np.argmin(m_total))
        rows.append({"station": station, "block": block,
                     "best start": best,
                     "E[min unavailable] at best": m_total[best],
                     "E[min unavailable] at half": m_total[cap // 2]})

        ax = axes[r, c]
        starts = np.arange(cap + 1)
        ax.plot(starts, m_empty, color="#2a78d6", lw=1.5, label="empty")
        ax.plot(starts, m_full, color="#d65a2a", lw=1.5, label="full")
        ax.plot(starts, m_total, color="0.2", lw=2, label="empty + full")
        ax.axvline(best, color="0.5", ls=":", lw=1)
        ax.set_title(f"{station} | {block} (best start: {best} bikes)", fontsize=10)
        ax.set_ylim(0, 300)
        ax.grid(axis="y", color="0.9")
        ax.spines[["top", "right"]].set_visible(False)
        if c == 0:
            ax.set_ylabel("Expected minutes unavailable")
        if r == len(stations) - 1:
            ax.set_xlabel("Bikes at start of block")

axes[0, 0].legend(fontsize=8, frameon=False)
fig.suptitle("Expected unavailable time per 300-minute block, by starting state (exact)")
fig.tight_layout()
fig.savefig(IMG / "unavailable_time_by_start.png", dpi=200, bbox_inches="tight")
plt.show()

print(pd.DataFrame(rows).round(1).to_string(index=False))

## F. Time until a station first becomes unavailable

For runs that reach 0 or capacity within the block, the first step at which this happens, converted to
minutes and a clock time. Only runs that become unavailable are included, so report it together with
the probability of becoming unavailable.

In [ ]:
rows = []
for station, cap in zip(stations, station_capacities):
    for block in ["AM", "PM"]:
        after = sims[station][block][:, 1:]
        hit = (after == 0) | (after == cap)
        ever = hit.any(axis=1)
        first_step = hit.argmax(axis=1) + 1                # first step at 0 or capacity
        T_min = first_step[ever] * STEP_MIN
        any_hit = ever.any()
        rows.append({"station": station, "block": block,
                     "P(unavailable in block)": ever.mean(),
                     "E[T | unavailable] (min)": T_min.mean() if any_hit else np.nan,
                     "median T (min)": np.median(T_min) if any_hit else np.nan,
                     "mean clock time": clock(block, T_min.mean()) if any_hit else "never"})

print(pd.DataFrame(rows).round(2).to_string(index=False))